In [ ]:
# ============================================================
# Imports & globale Konfiguration
# Imports & global configuration
# ============================================================

#TODO: Muss auf ECG Datensatz angepasst werden!


import numpy as np
import torch

from models.anoformer import (
    RANDOM_SEED,
    DEVICE,
    AnoFormerDataset,
    AnoFormerMainModel,
    set_seed,
)
from explainability.attention import AnoFormerExplainability

set_seed(RANDOM_SEED)
print(f"Active device: {DEVICE}")

In [ ]:
# ============================================================
# Demo-Beispiel (synthetische Zeitreihe mit Anomalien)
# Kompletter Durchlauf: Daten → Training → Erkennung → Visualisierung
# ============================================================

# ── 1. Synthetische Zeitreihe erzeugen ────────────────────────
RandomGenerator = np.random.default_rng(RANDOM_SEED)    # Reproduzierbarer Zufallsgenerator
TotalTime       = 2000                                   # 2000 Zeitschritte insgesamt
NumFeatures     = 3                                      # Drei parallele Feature-Kanäle

# Gleichmäßig verteilte Zeitachse von 0 bis 8π
TimeAxis = np.linspace(0, 8 * np.pi, TotalTime)

# Drei unterschiedliche Sinussignale + kleines Gauss-Rauschen
RawData  = np.column_stack([
    np.sin(TimeAxis)           + RandomGenerator.normal(0, 0.08, TotalTime),  # Feature 0: einfacher Sinus
    np.cos(1.5 * TimeAxis)     + RandomGenerator.normal(0, 0.06, TotalTime),  # Feature 1: Kosinus (höhere Freq.)
    0.5 * np.sin(3 * TimeAxis + 1) + RandomGenerator.normal(0, 0.1, TotalTime),  # Feature 2: Oberton
]).astype(np.float32)                                    # float32 für PyTorch-Kompatibilität

# Labels: 0 = normal, 1 = anomal (zu Beginn alles normal)
TrueLabels = np.zeros(TotalTime, dtype=np.int64)

# Fünf Anomalie-Segmente: Start- und End-Index jedes Segments
AnomalySegments = [
    (300, 320),    # Kurzer Spike zu Beginn
    (700, 730),    # Drift-Anomalie in der Mitte
    (1100, 1130),  # Amplitudenversatz
    (1500, 1540),  # Längere Multi-Spike-Anomalie
    (1800, 1820),  # Anomalie am Ende
]

for (Start, End) in AnomalySegments:
    # Zufälliger positiver Versatz zwischen 3 und 5 auf alle Features addieren
    RawData[Start:End, :] += RandomGenerator.uniform(3.0, 5.0, (End - Start, NumFeatures))
    TrueLabels[Start:End]  = 1                           # Bereich als anomal markieren

print(f"Datenpunkte gesamt : {TotalTime}")
print(f"Anomale Punkte     : {TrueLabels.sum()}")
print(f"Features           : {NumFeatures}")

# ── 2. Daten in Training und Test aufteilen ───────────────────
SplitPoint   = int(TotalTime * 0.75)     # 75% Training, 25% Test
TrainingData = RawData[:SplitPoint]       # Erste 1500 Punkte → Training (keine Anomalien in der Mehrheit)
TestData     = RawData[SplitPoint:]       # Letzte 500 Punkte → Test (enthält Anomalien am Ende)
TestLabels   = TrueLabels[SplitPoint:]   # Ground-Truth Labels nur für Testbereich

# ── 3. AnoFormerDataset-Objekte erstellen ─────────────────────
WindowSize      = 32                     # Jedes Fenster umfasst 32 Zeitschritte
TrainingDataset = AnoFormerDataset(
    TimeSeriesData    = TrainingData,
    WindowSize        = WindowSize,
    Stride            = 4,               # Stride 4 → überlappende Fenster (viele Trainingsdaten)
    NormalizationType = "zscore",         # Z-Score: Mittelwert 0, Std.-abw. 1
)
TestDataset = AnoFormerDataset(
    TimeSeriesData    = TestData,
    WindowSize        = WindowSize,
    Stride            = WindowSize,      # Stride = WindowSize → keine Überlappung beim Test
    NormalizationType = "zscore",
    Labels            = TestLabels,      # Labels für spätere Evaluation mitgeben
)

# DataLoader für Batching und Shuffling erstellen
TrainingLoader = TrainingDataset.CreateDataLoader(BatchSize=32, Shuffle=True)   # Mischen beim Training
TestLoader     = TestDataset.CreateDataLoader(BatchSize=32, Shuffle=False)      # Reihenfolge beim Test erhalten

print(f"\nTrainings-Fenster : {len(TrainingDataset)}")
print(f"Test-Fenster      : {len(TestDataset)}")

# ── 4. AnoFormer-Modell initialisieren ────────────────────────
AnoFormer = AnoFormerMainModel(
    NumFeatures               = NumFeatures,   # Muss mit den Daten übereinstimmen
    ModelDimension            = 32,            # Kleine Dimension für schnelles Demo-Training
    NumHeads                  = 2,             # Zwei Attention-Köpfe
    NumEncoderLayers          = 2,             # Flacher Encoder
    NumDecoderLayers          = 2,             # Flacher Decoder
    DiscriminatorLayers       = 2,             # Flacher Diskriminator
    ForwardDimension          = 64,            # FFN-Dimension = 2 × ModelDimension
    DropoutRate               = 0.1,           # 10% Dropout
    GeneratorLearningRate     = 5e-4,          # Etwas höhere LR für schnellere Konvergenz
    DiscriminatorLearningRate = 1e-3,          # Diskriminator lernt etwas schneller
    MaskingRate               = 0.15,          # 15% der Tokens zufällig maskiert
    AnomalyThreshold          = 0.75,          # 75. Perzentil für Schritt-2-Maskierung
    ReconstructionWeight      = 8.0,           # MSE-Loss 8× stärker gewichtet als GAN-Loss
)

# ── 5. Modell trainieren ──────────────────────────────────────
print("\n── Training beginnt ──")
AnoFormer.TrainModel(
    TrainingLoader,
    NumEpochs     = 30,   # 30 Epochen für Demo ausreichend
    PrintInterval = 5,    # Fortschritt alle 5 Epochen ausgeben
)

# ── 6. Anomalieerkennung auf Testdaten ────────────────────────
print("\n── Anomalieerkennung ──")
AnomalyScores, BinaryPredictions = AnoFormer.DetectAnomalies(TestLoader)
print(f"Erkannte anomale Fenster: {BinaryPredictions.sum()}")

# ── 7. Evaluation gegen Ground-Truth ─────────────────────────
print("\n── Evaluation ──")
# Window-Labels: ein Fenster gilt als anomal wenn mindestens ein Zeitschritt anomal ist
WindowLabels = np.array([
    TestLabels[i * WindowSize : (i + 1) * WindowSize].max()  # Max = 1 wenn irgendein Token = 1
    for i in range(len(TestDataset))
])
Metrics = AnoFormer.Evaluate(TestLoader, WindowLabels)  # Precision, Recall, F1 berechnen

# ── 8. Explainability-Visualisierungen ────────────────────────
print("\n── Explainability ──")
Explainer = AnoFormerExplainability(AnoFormer)  # Explainer mit Modell-Referenz instanziieren

# Trainingsverlauf aller Loss-Komponenten plotten
Explainer.ShowTrainingHistory()

# Ersten Test-Batch holen und auf Gerät schieben
FirstBatch, _ = next(iter(TestLoader))
FirstBatch     = FirstBatch.to(AnoFormer.Device)

# Rekonstruktion für diesen Batch berechnen (ohne Gradient)
with torch.no_grad():
    Masked, _     = AnoFormer.Masking(FirstBatch, None)    # Schritt-1-Maskierung
    Reconstructed = AnoFormer.Generator(Masked)            # Generator-Rekonstruktion

# Normierung rückgängig machen für lesbare Achsenwerte
OriginalNP      = TrainingDataset.InverseNormalization(FirstBatch[0].cpu().numpy())    # (T, F)
ReconstructedNP = TrainingDataset.InverseNormalization(Reconstructed[0].cpu().numpy()) # (T, F)

# Drei-Panel-Fehler-Visualisierung
Explainer.ShowReconstructionError(
    OriginalData      = OriginalNP,
    ReconstructedData = ReconstructedNP,
    AnomalyScores     = AnomalyScores,
    TrueLabels        = WindowLabels,
    Threshold         = AnoFormer.AnomalyLimit,   # Gelernter Schwellenwert verwenden
)

# Attention-Heatmap für das erste Fenster des ersten Batches
OneWindow = FirstBatch[:1]                         # Nur das erste Fenster: (1, T, F)
Explainer.ShowAttentionHeatmap(OneWindow, BlockIndex=0, HeadIndex=0, TitleSuffix="(Demo-Fenster)")

# Gradient×Input Feature-Wichtigkeit
Explainer.ShowFeatureImportance(OneWindow, FeatureNames=["Sinus", "Kosinus", "Mix"])

